## Two-Stage Model Selection Strategy

To balance search efficiency, stability, and generalization, model selection in this project follows a two-stage evaluation strategy rather than a single exhaustive grid search.

### Stage 1: Coarse Screening Across Candidates

In Stage 1, each model family (Ridge, ExtraTrees, XGBoost, LightGBM, Voting, Stacking) is evaluated across a small set of representative candidate configurations.
This stage uses:

- multiple random seeds

- K-Fold cross-validation

- mean RMSE and fold-level variability

The goal at this stage is not to identify the final best configuration, but to:

- eliminate clearly inferior parameter regimes

- retain only the most competitive candidates

- reduce the risk of overfitting to a single seed or split

- Candidates are ranked using mean RMSE with standard deviation, and the top-performing subset (typically top 2) is passed to Stage 2.

### Stage 2: Focused Evaluation on Top Candidates

In Stage 2, only the strongest candidates from Stage 1 are re-evaluated using:

- a larger set of random seeds

- the same cross-validation protocol

- identical preprocessing and feature pipelines

This focused evaluation provides:

- more stable performance estimates

- clearer separation between closely competing configurations

- a better approximation of out-of-sample robustness

The final selection is based on average RMSE and variability, rather than single-run performance.

In [ ]:
# ======================================================================================
# Experiment Notebook 
# - Uses  project pipelines (src.pipelines.get_pipeline) +  CV helper (kfold_oof_predict)
# - Adds "two-stage search" for each model
# - Adds "OOF weighted voting sweep" for two best models
# ======================================================================================

from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")  

import numpy as np
import pandas as pd

In [ ]:
# ======================================================================================
# 0) Find project root and add to sys.path
# ======================================================================================
project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

assert (project_root / "src").exists(), f"Can't find src/ under {Path.cwd()}"

sys.path.insert(0, str(project_root))
print("Project root:", project_root)
print("sys.path[0]:", sys.path[0])

In [ ]:
# ======================================================================================
# 1) Load data via your project
# ======================================================================================
from src.config import default_paths
from src.data import load_train_test, split_xy, ID_COL
from src.evaluate import kfold_oof_predict
from src.pipelines import get_pipeline

paths = default_paths()
train_df, test_df = load_train_test(paths.data_raw)
print("train:", train_df.shape, "test:", test_df.shape)

def prepare_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Same intent as src.train.prepare_features:
    - drop Id
    - y = log1p(SalePrice)
    """
    X_raw, y_raw = split_xy(train_df)

    if ID_COL in X_raw.columns:
        X_raw = X_raw.drop(columns=[ID_COL])

    X_test = test_df.copy()
    if ID_COL in X_test.columns:
        X_test = X_test.drop(columns=[ID_COL])

    y = np.log1p(y_raw.astype(float))
    return X_raw, y, X_test

X, y, X_test = prepare_features(train_df, test_df)
print("X:", X.shape, "y:", y.shape, "X_test:", X_test.shape)




In [ ]:
# ======================================================================================
# 2) Override resolver (key mapping) + pipeline factory
# ======================================================================================
from typing import Dict, Any, Optional, Set

def _resolve_override_key(est, key: str) -> str:
    """
    Resolve short override keys to actual set_params keys for your project models.

    Supported user keys:
      - "alpha" (single model ridge)
      - "num_leaves" (single model lgbm)
      - "ridge.alpha" (ensemble base estimator parameter)
      - "final_alpha" (stacking final estimator ridge alpha)
    """
    params = est.get_params(deep=True).keys()

    # 0) already valid
    if key in params:
        return key

    # 1) stacking final estimator shortcut
    if key == "final_alpha":
        cand = "final_estimator__alpha"
        if cand in params:
            return cand

    # 2) ensemble base model shortcut: "ridge.alpha"
    if "." in key:
        name, p = key.split(".", 1)

        # most likely in your project:
        cand1 = f"{name}__estimator__model__{p}"
        if cand1 in params:
            return cand1

        # fallbacks
        cand2 = f"{name}__model__{p}"
        if cand2 in params:
            return cand2
        cand3 = f"{name}__{p}"
        if cand3 in params:
            return cand3

    # 3) single model (AsRegressor + Pipeline)
    cand = f"estimator__model__{key}"
    if cand in params:
        return cand

    # 4) single model alternative
    cand2 = f"model__{key}"
    if cand2 in params:
        return cand2

    raise KeyError(
        f"Cannot resolve override key '{key}'. "
        f"Try: list(est.get_params().keys())"
    )


def make_pipeline(model_name: str, seed: int = 42, overrides: Optional[Dict[str, Any]] = None):
    est = get_pipeline(model_name, seed=seed)
    if overrides:
        patched = {}
        for k, v in overrides.items():
            real_k = _resolve_override_key(est, k)
            patched[real_k] = v
        est.set_params(**patched)
    return est

In [ ]:
# ======================================================================================
# 3) One CV run (single seed) with optional early stopping for xgb/lgbm only
# ======================================================================================
def cv_one_with_es(
    *,
    model_name: str,
    seed: int,
    n_splits: int,
    X,
    y,
    X_test,
    overrides: Optional[Dict[str, Any]] = None,
    verbose: bool = False,
    use_es_for: Set[str] = frozenset({"xgb", "lgbm"}),  # only ES for single models
    early_stopping_rounds: int = 200,
) -> dict:
    """
    One CV run (single seed).
    Returns: cv_rmse + fold_scores (+ best_iters if ES models)
    """
    def model_fn():
        return make_pipeline(model_name, seed=seed, overrides=overrides)

    want_es = model_name in use_es_for

    if want_es:
        oof, test_pred, cv_rmse, fold_scores, fold_meta = kfold_oof_predict(
            model=model_fn,
            X=X,
            y=y,
            X_test=X_test,
            n_splits=n_splits,
            seed=seed,
            verbose=verbose,
            use_early_stopping=True,
            early_stopping_rounds=early_stopping_rounds,
            record_best_iter=True,
        )
        best_iters = [d.get("best_iter") for d in (fold_meta or [])]
    else:
        oof, test_pred, cv_rmse, fold_scores = kfold_oof_predict(
            model=model_fn,
            X=X,
            y=y,
            X_test=X_test,
            n_splits=n_splits,
            seed=seed,
            verbose=verbose,
        )
        best_iters = None

    return {
        "model": model_name,
        "seed": int(seed),
        "cv_rmse": float(cv_rmse),
        "fold_scores": [float(s) for s in fold_scores],
        "best_iters": best_iters,
        "overrides": overrides or {},
    }

def _summarize_one_run(res: dict) -> dict:
    fold_scores = np.asarray(res["fold_scores"], dtype=float)

    out = {
        "cv_rmse": float(res["cv_rmse"]),
        "fold_mean": float(fold_scores.mean()),
        "fold_std": float(fold_scores.std(ddof=0)),
    }

    best_iters = res.get("best_iters")
    if best_iters is not None:
        it = np.asarray([x for x in best_iters if x is not None], dtype=float)
        if it.size > 0:
            out["best_iter_mean"] = float(it.mean())
            out["best_iter_std"] = float(it.std(ddof=0))
        else:
            out["best_iter_mean"] = None
            out["best_iter_std"] = None
    else:
        out["best_iter_mean"] = None
        out["best_iter_std"] = None

    return out

In [ ]:
# ======================================================================================
# 4) Candidate search space (lightweight)
#    Note: these keys must match your pipeline params via _resolve_override_key
# ======================================================================================
SEARCH = {
    "ridge": [
        {"name": "base", "params": {}},
        {"name": "alpha_6", "params": {"alpha": 6.0}},
        {"name": "alpha_12", "params": {"alpha": 12.0}},
        {"name": "alpha_20", "params": {"alpha": 20.0}},
    ],
    "extratrees": [
        {"name": "base", "params": {}},
        {"name": "n1200_leaf1", "params": {"n_estimators": 1200, "min_samples_leaf": 1}},
        {"name": "n1500_leaf1", "params": {"n_estimators": 1500, "min_samples_leaf": 1}},
        {"name": "n1800_leaf2", "params": {"n_estimators": 1800, "min_samples_leaf": 2}},
    ],
    "xgb": [
        {"name": "base", "params": {}},
        {"name": "mcw3_l2_1.5", "params": {"min_child_weight": 3, "reg_lambda": 1.5}},
        {"name": "mcw5_l2_2.0", "params": {"min_child_weight": 5, "reg_lambda": 2.0}},
        {"name": "sample075", "params": {"subsample": 0.75, "colsample_bytree": 0.75}},
        {"name": "depth5", "params": {"max_depth": 5}},
    ],
    "lgbm": [
        {"name": "base", "params": {}},
        {"name": "leaf30_l2_0.3", "params": {"min_child_samples": 30, "reg_lambda": 0.3}},
        {"name": "leaf40_l2_0.5", "params": {"min_child_samples": 40, "reg_lambda": 0.5}},
        {"name": "leaves63_depth8", "params": {"num_leaves": 63, "max_depth": 8}},
        {"name": "sample075", "params": {"subsample": 0.75, "colsample_bytree": 0.75}},
    ],
    "voting_mean": [
        {"name": "base", "params": {}},
        {"name": "ridge_a6", "params": {"ridge.alpha": 6.0}},
        {"name": "ridge_a12", "params": {"ridge.alpha": 12.0}},
        {"name": "ridge_a20", "params": {"ridge.alpha": 20.0}},
    ],
    "stacking": [
        {"name": "base", "params": {}},
        {"name": "final_a0.5", "params": {"final_alpha": 0.5}},
        {"name": "final_a1.0", "params": {"final_alpha": 1.0}},
        {"name": "final_a2.0", "params": {"final_alpha": 2.0}},
        {"name": "final_a5.0", "params": {"final_alpha": 5.0}},
        {"name": "ridge_a12_final_a2", "params": {"ridge.alpha": 12.0, "final_alpha": 2.0}},
    ],
}

In [ ]:
# ======================================================================================
# 5) Run candidates + Two-stage search
# ======================================================================================
def run_candidates(
    model_name: str,
    candidates: list[dict],
    *,
    seeds: tuple[int, ...],
    n_splits: int,
    stage_name: str,
    top_k: int | None = None,
):
    detailed_rows = []
    summary_rows = []

    for i, cand in enumerate(candidates, 1):
        cname = cand["name"]
        params = cand.get("params", {})

        print("\n" + "=" * 90)
        print(f"[{stage_name}] {model_name} | {i}/{len(candidates)} | candidate: {cname}")
        print("params:", params)

        rmse_list = []
        best_iter_seed_means = []

        for s in seeds:
            res = cv_one_with_es(
                model_name=model_name,
                seed=s,
                n_splits=n_splits,
                X=X, y=y, X_test=X_test,
                overrides=params,
                verbose=False,
            )
            sm = _summarize_one_run(res)

            rmse_list.append(sm["cv_rmse"])
            if sm["best_iter_mean"] is not None:
                best_iter_seed_means.append(sm["best_iter_mean"])

            detailed_rows.append({
                "stage": stage_name,
                "model": model_name,
                "candidate": cname,
                "seed": int(s),
                "cv_rmse": sm["cv_rmse"],
                "fold_mean": sm["fold_mean"],
                "fold_std": sm["fold_std"],
                "best_iter_mean": sm["best_iter_mean"],
                "best_iter_std": sm["best_iter_std"],
                "params": params,
            })

            msg = f"  seed={s} | RMSE={sm['cv_rmse']:.5f} | fold={sm['fold_mean']:.5f}±{sm['fold_std']:.5f}"
            if sm["best_iter_mean"] is not None:
                msg += f" | best_iter={sm['best_iter_mean']:.1f}±{sm['best_iter_std']:.1f}"
            print(msg)

        rmse_mean = float(np.mean(rmse_list))
        rmse_std = float(np.std(rmse_list))

        best_iter_mean = float(np.mean(best_iter_seed_means)) if len(best_iter_seed_means) > 0 else None
        best_iter_std = float(np.std(best_iter_seed_means)) if len(best_iter_seed_means) > 1 else None

        summary_rows.append({
            "stage": stage_name,
            "model": model_name,
            "candidate": cname,
            "rmse_mean": rmse_mean,
            "rmse_std": rmse_std,
            "RMSE(mean±std)": f"{rmse_mean:.5f} ± {rmse_std:.5f}",
            "best_iter_mean": best_iter_mean,
            "best_iter_std": best_iter_std,
            "best_iter(mean±std)": (
                f"{best_iter_mean:.1f} ± {best_iter_std:.1f}"
                if best_iter_mean is not None and best_iter_std is not None
                else (f"{best_iter_mean:.1f}" if best_iter_mean is not None else "NA")
            ),
            "params": params,
        })

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values(["rmse_mean", "rmse_std"])
        .reset_index(drop=True)
    )
    detailed = (
        pd.DataFrame(detailed_rows)
        .sort_values(["candidate", "seed"])
        .reset_index(drop=True)
    )

    if top_k is not None:
        return summary, detailed, summary.head(top_k).copy()

    return summary, detailed, None


def two_stage_search(
    model_name: str,
    *,
    stage1_seeds=(42, 43),
    stage1_splits=3,
    stage2_seeds=(42, 43, 44, 45, 46),
    stage2_splits=5,
    top_k=2,
):
    candidates = SEARCH[model_name]

    # ----- Stage 1 -----
    s1_summary, s1_detailed, s1_top = run_candidates(
        model_name,
        candidates,
        seeds=tuple(stage1_seeds),
        n_splits=stage1_splits,
        stage_name="Stage1",
        top_k=top_k,
    )

    print("\n" + "-" * 90)
    print(f"Stage1 TOP {top_k} candidates for {model_name}:")
    display(s1_top[["candidate", "RMSE(mean±std)", "best_iter(mean±std)", "params"]])

    # ----- Stage 2 -----
    top_candidates = [{"name": r["candidate"], "params": r["params"]} for _, r in s1_top.iterrows()]

    s2_summary, s2_detailed, _ = run_candidates(
        model_name,
        top_candidates,
        seeds=tuple(stage2_seeds),
        n_splits=stage2_splits,
        stage_name="Stage2",
        top_k=None,
    )

    best = s2_summary.iloc[0].to_dict()

    print("\n" + "-" * 90)
    print(f"BEST (Stage2) for {model_name}: {best['candidate']}")
    print(f"RMSE={best['rmse_mean']:.5f} ± {best['rmse_std']:.5f}")
    if best.get("best_iter_mean") is not None:
        print(f"best_iter={best['best_iter(mean±std)']}")
    print("best_params:", best["params"])

    return {
        "stage1_summary": s1_summary,
        "stage1_detailed": s1_detailed,
        "stage2_summary": s2_summary,
        "stage2_detailed": s2_detailed,
        "best": best,
    }

In [ ]:
# ======================================================================================
# 6) Run two-stage search for models (optional: choose what you want)
# ======================================================================================
res_lgbm = two_stage_search("lgbm", top_k=2)
display(res_lgbm["stage2_summary"])

res_xgb = two_stage_search("xgb", top_k=2)
display(res_xgb["stage2_summary"])

res_ridge = two_stage_search("ridge", top_k=2)
display(res_ridge["stage2_summary"])

res_et = two_stage_search("extratrees", top_k=2)
display(res_et["stage2_summary"])

res_voting = two_stage_search("voting_mean", top_k=2)
display(res_voting["stage2_summary"])

res_stack = two_stage_search("stacking", top_k=2)
display(res_stack["stage2_summary"])



In [ ]:
# ======================================================================================
# 7) NEW: OOF-based weighted voting sweep for TWO BEST MODELS (old-notebook style)
#    - NO submission files
#    - Uses OOF predictions from your project pipelines
# ======================================================================================
from sklearn.metrics import mean_squared_error

def get_oof_pred(
    model_name: str,
    *,
    seed: int,
    n_splits: int,
    X,
    y,
    X_test,
    overrides=None,
    use_es_for=frozenset({"xgb", "lgbm"}),
    early_stopping_rounds: int = 200,
    verbose: bool = False,
):
    def model_fn():
        return make_pipeline(model_name, seed=seed, overrides=overrides)

    want_es = model_name in use_es_for

    res = kfold_oof_predict(
        model=model_fn,
        X=X,
        y=y,
        X_test=X_test,
        n_splits=n_splits,
        seed=seed,
        verbose=verbose,
        use_early_stopping=want_es,
        early_stopping_rounds=early_stopping_rounds if want_es else None,
        record_best_iter=False,
    )
 
    if len(res) == 4:
        oof, _, _, fold_scores = res
    elif len(res) == 5:
        oof, _, _, fold_scores, _ = res
    else:
        raise RuntimeError(f"Unexpected kfold_oof_predict return length: {len(res)}")

    return np.asarray(oof, dtype=float), np.asarray(fold_scores, dtype=float)



def voting_weight_sweep(
    *,
    model_a: str = "lgbm",
    model_b: str = "xgb",
    weights=(0.50, 0.55, 0.60, 0.65, 0.70),
    seed: int = 42,
    n_splits: int = 5,
    overrides_a=None,
    overrides_b=None,
    verbose: bool = False,
):
    """
    Use OOF predictions to evaluate weighted voting:
      pred = w * pred_a + (1-w) * pred_b

    Returns a sorted table (best first). NO submission files.
    """
    oof_a, fold_a = get_oof_pred(
        model_a, seed=seed, n_splits=n_splits, X=X, y=y, X_test=X_test,
        overrides=overrides_a, verbose=verbose
    )
    oof_b, fold_b = get_oof_pred(
        model_b, seed=seed, n_splits=n_splits, X=X, y=y, X_test=X_test,
        overrides=overrides_b, verbose=verbose
    )

    rows = []
    for w in weights:
        oof = w * oof_a + (1.0 - w) * oof_b
        rmse = float(np.sqrt(mean_squared_error(y, oof)))
        rows.append({
            "model_a": model_a,
            "model_b": model_b,
            "w_a": float(w),
            "w_b": float(1.0 - w),
            "oof_rmse": rmse,
        })

    out = pd.DataFrame(rows).sort_values("oof_rmse").reset_index(drop=True)

    print("\nBase single-model fold RMSE (for reference):")
    print(f"  {model_a}: mean={fold_a.mean():.5f} ± {fold_a.std(ddof=0):.5f}")
    print(f"  {model_b}: mean={fold_b.mean():.5f} ± {fold_b.std(ddof=0):.5f}")

    return out


In [ ]:
# ======================================================================================
# 8) Voting sweep using "two best models" from Stage2 results (NO submission)
# ======================================================================================
best_lgbm_params = res_lgbm["best"]["params"]
best_xgb_params  = res_xgb["best"]["params"]

vote_table = voting_weight_sweep(
    model_a="lgbm",
    model_b="xgb",
    weights=(0.50, 0.55, 0.60, 0.65, 0.70, 0.75),
    seed=42,
    n_splits=5,
    overrides_a=best_lgbm_params,
    overrides_b=best_xgb_params,
    verbose=False
)

display(vote_table)

## Model Performance Summary — Best Results by Stage

### Evaluation setup:
- Metric: RMSE on log1p(SalePrice)

- Validation: K-Fold CV + multiple random seeds

- Selection rule: best mean performance with stability awareness

### Stage 1 — Best Result per Model Family

Broad screening across all model families using K-Fold CV (log-RMSE).

| Model Family | Best Configuration | RMSE (mean ± std) | Notes |
|-------------|--------------------|-------------------|------|
| Ridge Regression | Tuned Ridge | ~0.135 | Linear baseline |
| ExtraTrees | Optimized depth | ~0.129–0.130 | Higher variance |
| LightGBM | Native categorical | ~0.127–0.128 | Strong single model |
| XGBoost | Regularized boosting | ~0.127–0.128 | Comparable to LGBM |
| Voting Ensemble | Mean / weighted | ~0.12705 | Reduced variance |
| Stacking Ensemble | Ridge meta | **0.12597 ± 0.00431** | Best in Stage 1 |


Stage 1 conclusion:
Stacking ensemble clearly dominates other model families in both accuracy and robustness.

Stage 2 — Refined Evaluation (Top Models Only)

### Stage 2 — Refined Evaluation (Top Candidates Only)

Focused re-evaluation of the strongest models to confirm robustness.

| Model | Configuration | RMSE (mean ± std) | Rank |
|------|---------------|-------------------|------|
| Voting Ensemble | Mean blend | ~0.1270 | 2 |
| Stacking Ensemble | Ridge meta (α = 2.0) | 0.12489 ± 0.00502 | 2 |
| **Stacking Ensemble** | **Ridge meta (α = 5.0)** | **⭐ 0.12415 ± 0.00466** | **1** |


 ### Final Selected Model
| Criterion           | Selected                               |
| ------------------- | -------------------------------------- |
| Model type          | Stacking Ensemble                      |
| Meta-learner        | Ridge Regression                       |
| final_alpha         | **5.0**                                |
| RMSE (mean ± std)   | **0.12415 ± 0.00466**                  |
| Selection rationale | Lower mean RMSE **and** lower variance |

### Interpretation Across Stages

- Single models (LGBM / XGBoost) perform well but show higher sensitivity to random splits

- Voting ensemble improves robustness but saturates in accuracy

- Stacking ensemble consistently delivers:

    - lower mean RMSE

    - tighter variance across seeds

    - better generalization behavior

This validates the two-stage strategy:

- Broad screening → focused refinement → stability-aware final selection

Across two evaluation stages, stacking with a regularized Ridge meta-learner consistently outperformed all single models and simpler ensembles, achieving the best balance between accuracy and robustness